# E1.1 — Does cross-modal correspondence need a similarity-shaped space?

**The matched pair.** E1 mapped DINOv2 into bge-m3, a text encoder trained
contrastively on 1.2 billion text pairs — a space *built* for cosine
geometry. This notebook repeats the identical experiment with one variable
changed: the text encoder becomes **GPT-2, mean-pooled**, trained only on
next-token prediction and never optimized for similarity.

Everything else is held fixed: same images, same captions, same DINOv2
settings, same split, same ridge and alpha sweep, same shuffle control,
same pre-registered bands.

**Why this pair.** GPT-2's representations are severely anisotropic —
measured in Experiment A at an effective dimensionality of 2.2 to 10.9 out
of 768, variance collapsed into a handful of directions. bge-m3's are
shaped so that related texts point the same way. The pair therefore
isolates one question: *does the cross-modal correspondence require a
similarity-shaped target space, or does it exist in raw language-model
representations too?*

**Pre-registered predictions** (fixed before running):

| Outcome | Interpretation |
|---|---|
| R² holds, R@1 collapses | correspondence exists in raw LM representations, but *usable* retrieval geometry needs similarity training — ridge absorbs anisotropy, cosine cannot |
| both hold | convergence does not depend on similarity training at all — the strongest result |
| both collapse | bge-m3's contrastive shaping was doing much of the work in E1; the cross-modal claim is correspondingly qualified |
| R@1 holds, R² collapses | unexpected; suspect a bug before interpreting |

The first is the expected outcome, and it follows directly from this
project's own Procrustes-versus-ridge finding: a fit with per-direction
scaling can undo anisotropy, an angle-only metric cannot.

**Reference numbers to beat or fall short of (E1, bge-m3):**
held-out R² **0.511**, cosine **0.889**, R@1 **0.358**, R@5 **0.706**,
shuffle gap **0.641**, ceiling-matched share **40.9%**.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
!pip -q install torch torchvision transformers pillow

In [ ]:
import numpy as np, torch, json, io, zipfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

DATA_DIR = Path(os.environ["DATA_DIR"]); DATA_DIR.mkdir(exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(0)

IMG_MODEL   = "facebook/dinov2-base"
TEXT_ENCODER = "gpt2"          # "gpt2" (this run) or "bge-m3" (the E1 arm)

# MATCHED-PAIR CONSTRAINT: the image side must be configured exactly as
# the run that produced the E1 reference numbers below - which was
# CLS-only pooling at N_PAIRS = 8000 with averaged captions. If E1 is
# re-run with cls+patch, change BOTH settings here AND replace the
# reference dict in the evaluation cell, or the comparison is not a
# matched pair.
POOLING      = "cls"
MAX_LEN      = 64              # same truncation as Experiment A
ALL_CAPTIONS = True            # average all five, as in E1

N_EVAL     = 1000
N_PAIRS    = 8000              # pinned to the E1 reference run
ALPHA_GRID = [1e-3, 1e-2, 1e-1, 1.0, 10.0]

_W = {"small": 384, "base": 768, "large": 1024, "giant": 1536}
_w = _W[[k for k in _W if k in IMG_MODEL][0]]
if POOLING == "cls+patch":
    _w *= 2

print(f"text encoder: {TEXT_ENCODER}")
print(f"image side: {IMG_MODEL}, pooling {POOLING} -> width {_w}")
print(f"N_PAIRS {N_PAIRS} -> {(N_PAIRS - N_EVAL) / _w:.1f} rows per "
      f"input dim (need >= 5)")
print("device:", DEV)

## 1. Data — identical to E1

In [ ]:
ANN = DATA_DIR / "annotations_trainval2017.zip"
if not ANN.exists():
    urllib.request.urlretrieve(
        "http://images.cocodataset.org/annotations/"
        "annotations_trainval2017.zip", str(ANN))
with zipfile.ZipFile(ANN) as z:
    with z.open("annotations/captions_train2017.json") as f:
        ann = json.load(f)

caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
first_cap = {k: (v if ALL_CAPTIONS else v[:1]) for k, v in caps.items()}
url = {im["id"]: im["coco_url"] for im in ann["images"]}
ids = sorted(set(first_cap) & set(url))[:N_PAIRS]
captions = [first_cap[i] for i in ids]
print(f"{len(ids)} image/caption groups")

## 2. Image side — DINOv2, identical to E1 (cached and resumable)

In [ ]:
from transformers import AutoImageProcessor, AutoModel

WORKERS, CHUNK, BATCH = 32, 512, 64
CKPT = DATA_DIR / f"e11_img_ckpt_{POOLING}.npz"

proc = AutoImageProcessor.from_pretrained(IMG_MODEL)
vis  = AutoModel.from_pretrained(IMG_MODEL).to(DEV).eval()
if DEV == "cuda":
    vis = vis.half()

def fetch(n):
    try:
        with urllib.request.urlopen(url[ids[n]], timeout=8) as r:
            return n, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return n, None

if CKPT.exists():
    d = np.load(str(CKPT))
    img_list, keep, start = [d["img"]], list(d["keep"]), int(d["next"])
    print(f"resuming at row {start} ({len(keep)} encoded)")
else:
    img_list, keep, start = [], [], 0

import time; t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for c0 in range(start, len(ids), CHUNK):
        c1 = min(c0 + CHUNK, len(ids))
        got = sorted((r for r in pool.map(fetch, range(c0, c1))
                      if r[1] is not None), key=lambda x: x[0])
        for b0 in range(0, len(got), BATCH):
            batch = got[b0:b0 + BATCH]
            with torch.no_grad():
                x = proc(images=[im for _, im in batch],
                         return_tensors="pt").to(DEV)
                if DEV == "cuda":
                    x["pixel_values"] = x["pixel_values"].half()
                o = vis(**x).last_hidden_state
                h = (o[:, 0] if POOLING == "cls"
                     else torch.cat([o[:, 0], o[:, 1:].mean(1)], dim=-1))
            img_list.append(h.float().cpu().numpy())
            keep += [n for n, _ in batch]
        r = (c1 - start) / max(time.time() - t0, 1e-9)
        print(f"  {c1}/{len(ids)} kept {len(keep)} {r:.0f} img/s "
              f"ETA {(len(ids)-c1)/max(r,1e-9)/60:.1f} min")
        np.savez_compressed(str(CKPT),
                            img=np.concatenate(img_list).astype(np.float32),
                            keep=np.array(keep), next=c1)
IMG = np.concatenate(img_list).astype(np.float64)
print("image embeddings:", IMG.shape)

## 3. Text side — the one variable

GPT-2 with attention-masked mean pooling, exactly the procedure Experiment
A used on WikiText. Captions are averaged across the five per image when
`ALL_CAPTIONS`, matching the E1 arm.

In [ ]:
if TEXT_ENCODER == "gpt2":
    from transformers import AutoTokenizer, AutoModel as HFAutoModel
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    lm = HFAutoModel.from_pretrained("gpt2").to(DEV).eval()

    def encode_text(strings, bs=64):
        out = []
        for b in range(0, len(strings), bs):
            enc = tok(strings[b:b + bs], return_tensors="pt", padding=True,
                      truncation=True, max_length=MAX_LEN).to(DEV)
            with torch.no_grad():
                h = lm(**enc).last_hidden_state          # [B, T, 768]
            m = enc["attention_mask"].unsqueeze(-1).float()
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)   # masked mean
            out.append(pooled.float().cpu().numpy())
        return np.concatenate(out).astype(np.float64)
else:
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer("BAAI/bge-m3", device=DEV)
    def encode_text(strings, bs=64):
        return _st.encode(strings, batch_size=bs, convert_to_numpy=True,
                          show_progress_bar=True).astype(np.float64)

flat, owner = [], []
for j, k in enumerate(keep):
    for cap in (captions[k] if isinstance(captions[k], list)
                else [captions[k]]):
        flat.append(cap); owner.append(j)
owner = np.array(owner)
E = encode_text(flat)
TXT = np.stack([E[owner == j].mean(0) for j in range(len(keep))])
print(f"{len(flat)} captions -> {TXT.shape} averaged targets "
      f"({len(flat)/len(keep):.1f} per image)")
assert len(IMG) == len(TXT), "row alignment broken"
np.savez_compressed(str(DATA_DIR / f"crossmodal_pairs_{TEXT_ENCODER}.npz"),
                    img=IMG.astype(np.float32), txt=TXT.astype(np.float32))

## 4. The anisotropy of the target space — the mechanism under test

Measured before fitting, so the prediction is anchored to a number rather
than an assumption. Experiment A found GPT-2's effective dimensionality at
2.2 to 10.9 of 768; bge-m3 should be far higher.

In [ ]:
def spectrum_report(M, name):
    Mc = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(Mc, full_matrices=False, compute_uv=False)
    p = s**2 / (s**2).sum()
    eff = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum()))
    Mn = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
    i2 = rng.choice(len(M), 2000); j2 = rng.choice(len(M), 2000)
    k = i2 != j2
    pc = float((Mn[i2[k]] * Mn[j2[k]]).sum(1).mean())
    print(f"  {name:22s} eff.rank {eff:7.1f} of {M.shape[1]:5d}   "
          f"PC1 {p[0]:.3f}   mean pair-cos {pc:+.3f}")
    return eff, p[0], pc

print("target-space geometry:")
spectrum_report(TXT, f"TXT ({TEXT_ENCODER})")
spectrum_report(IMG, "IMG (DINOv2)")
print("\nlow effective rank + high pair-cosine = anisotropic;")
print("ridge can absorb this, cosine similarity cannot")

## 5. Fit and evaluate — identical protocol to E1

In [ ]:
def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

def ridge(X, Y, a):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

def recall(S):
    order = np.argsort(-S, axis=1)
    r = (order == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

d_img = IMG.shape[1]
idx = rng.permutation(len(IMG))
te, tr = idx[:N_EVAL], idx[N_EVAL:]
POWERED = len(tr) / d_img >= 5
print(f"rows/dim {len(tr)/d_img:.1f} -> "
      f"{'POWERED' if POWERED else 'UNDERPOWERED'}")

nv = max(200, len(tr) // 5)
vtr, vva = tr[:-nv], tr[-nv:]
best = (None, -9)
for a in ALPHA_GRID:
    Pv = IMG[vva] @ ridge(IMG[vtr], TXT[vtr], a)
    sc = 1 - ((TXT[vva]-Pv)**2).sum()/((TXT[vva]-TXT[vva].mean(0))**2).sum()
    print(f"  alpha {a:<7g} validation R2 {sc:.3f}")
    if sc > best[1]:
        best = (a, sc)
ALPHA_BEST = best[0]
print(f"  -> alpha {ALPHA_BEST}\n")

W = ridge(IMG[tr], TXT[tr], ALPHA_BEST)
P = IMG[te] @ W
r2 = 1 - ((TXT[te]-P)**2).sum()/((TXT[te]-TXT[te].mean(0))**2).sum()
cos = float((l2n(P) * l2n(TXT[te])).sum(1).mean())
Ws = ridge(IMG[tr], TXT[tr][rng.permutation(len(tr))], ALPHA_BEST)
r2s = 1 - ((TXT[te]-IMG[te]@Ws)**2).sum()/((TXT[te]-TXT[te].mean(0))**2).sum()

gal = l2n(TXT[te])
r_adapt = recall(l2n(P) @ gal.T)
raw = np.zeros_like(TXT[te]); m = min(d_img, TXT.shape[1])
raw[:, :m] = IMG[te][:, :m]
r_raw = recall(l2n(raw) @ gal.T)

# measured by E1 with POOLING='cls', N_PAIRS=8000, averaged captions
E1 = dict(r2=0.511, cos=0.889, r1=0.358, r5=0.706, gap=0.641)
print(f"held-out ridge : R2 {r2:.3f}   cosine {cos:.3f}    "
      f"[E1/bge-m3: {E1['r2']:.3f} / {E1['cos']:.3f}]")
print(f"shuffle control: R2 {r2s:.3f}   gap {r2-r2s:.3f}   "
      f"[E1 gap {E1['gap']:.3f}]")
print("retrieval image->text: " +
      "  ".join(f"R@{k}={v:.3f}" for k, v in r_adapt.items()) +
      f"    [E1: R@1={E1['r1']:.3f} R@5={E1['r5']:.3f}]")
print("raw (chance floor)   : " +
      "  ".join(f"R@{k}={v:.3f}" for k, v in r_raw.items()))

keep_r2 = r2 / E1["r2"]; keep_r1 = r_adapt[1] / E1["r1"]
print(f"\nretained vs the bge-m3 arm:  R2 {keep_r2:.0%}   "
      f"R@1 {keep_r1:.0%}")
if not POWERED:
    print("\nVERDICT WITHHELD - underpowered")
elif r2 - r2s < 0.2:
    print("\nVERDICT: alignment check FAILED - suspect row misalignment")
elif keep_r2 > 0.6 and keep_r1 > 0.6:
    print("\nVERDICT: BOTH HOLD - convergence does not depend on "
          "similarity training")
elif keep_r2 > 0.6 and keep_r1 < 0.35:
    print("\nVERDICT: R2 HOLDS, RETRIEVAL COLLAPSES - the predicted "
          "split. The correspondence exists in raw language-model")
    print("representations; usable retrieval geometry needs similarity "
          "training. Ridge absorbs anisotropy, cosine cannot.")
elif keep_r2 < 0.35:
    print("\nVERDICT: BOTH COLLAPSE - bge-m3's contrastive shaping was "
          "doing much of the work in E1; qualify the cross-modal claim")
else:
    print("\nVERDICT: INTERMEDIATE - report the retained fractions and "
          "the target-space spectra together")

## 6. Reading the pair

The two arms differ in exactly one component, so any gap is attributable
to the text encoder's geometry. Report them side by side, together with
the measured effective ranks from section 4 — the mechanism and the
outcome then appear in the same table, which is what turns a comparison
into an explanation.

If retrieval collapses while R² survives, the finding is specific and
worth stating plainly: **convergence is a property of what models learn;
whether it is usable depends on how the space is shaped.** That separates
two things usually conflated, and it follows the same logic as this
project's Procrustes result — a metric that cannot rescale directions
fails where a fit that can succeeds.